# 02 Ground Truth Coordinate Generator

This notebook rebuilds the approved test set from the current manually annotated images.

The process is:

1. Read the current images in `examples/ground_truth_annotated/`.
2. Match each annotation to its raw image in `examples/residence_images/`.
3. Detect the green box drawn with `#43b65e`.
4. Scale coordinates only when the annotated and raw image dimensions differ.
5. Write the final ground truth CSV and a new benchmark based on the current filenames.
6. Generate confirmation screenshots and a contact sheet.

It does not read `day3_selected_frames.csv` and it does not use manual overrides.


In [ ]:
# Imports.
# Pillow is not reinstalled because doing so inside an active runtime can create
# a mixed PIL installation. Use a fresh Colab runtime if PIL was previously modified.

from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
import numpy as np
import pandas as pd
import shutil
import math
import re

try:
    import cv2
    CV2_AVAILABLE = True
except Exception:
    CV2_AVAILABLE = False

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print('Drive is already mounted or this is not a Colab runtime.')

print('Pillow version:', Image.__version__)
print('OpenCV available:', CV2_AVAILABLE)


In [ ]:
# Project paths.

PROJECT_ROOT = Path('/content/drive/MyDrive/autonomous-delivery-robot/modules/perception/visual_grounding')

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'Project folder not found: {PROJECT_ROOT}. Update PROJECT_ROOT above.')

EXAMPLES_DIR = PROJECT_ROOT / 'examples'
ANNOTATED_DIR = EXAMPLES_DIR / 'ground_truth_annotated'
RAW_IMAGE_DIR = EXAMPLES_DIR / 'residence_images'

GT_CSV = EXAMPLES_DIR / 'ground_truth_boxes.csv'
GROUNDING_BENCHMARK_CSV = EXAMPLES_DIR / 'grounding_benchmark_from_ground_truth.csv'
PARSED_BENCHMARK_CSV = EXAMPLES_DIR / 'grounding_benchmark_with_task_parser.csv'

OUTPUT_DIR = PROJECT_ROOT / 'results' / 'ground_truth_coordinate_update'
FINAL_REVIEW_SCREENSHOT_DIR = OUTPUT_DIR / 'final_ground_truth_review_screenshots'
CONTACT_SHEET_PATH = OUTPUT_DIR / 'final_ground_truth_review_contact_sheet.jpg'
SKIPPED_CSV = OUTPUT_DIR / 'skipped_ground_truth_images.csv'
MATCH_AUDIT_CSV = OUTPUT_DIR / 'annotation_to_raw_image_match_audit.csv'

GREEN_HEX = '#43b65e'
GREEN_TOLERANCE_STRICT = 8
GREEN_TOLERANCE_LOOSE = 35
MIN_GREEN_PIXELS = 20
IMAGE_EXTENSIONS = ['.jpg', '.jpeg', '.png', '.webp']

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_REVIEW_SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Annotated images:', ANNOTATED_DIR)
print('Raw residence images:', RAW_IMAGE_DIR)
print('Ground truth CSV:', GT_CSV)
print('Benchmark CSV:', GROUNDING_BENCHMARK_CSV)


In [ ]:

# Helper functions.


def hex_to_rgb(hex_code):
    hex_code = str(hex_code).strip().replace('#', '')
    return tuple(int(hex_code[i:i + 2], 16) for i in (0, 2, 4))


def list_images(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS])


def normalize_image_id(path_or_name):
    stem = Path(path_or_name).stem
    stem = re.sub(r'^(annotated_|gt_|ground_truth_)', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'(_annotated|_gt|_ground_truth|_with_box|_boxed)$', '', stem, flags=re.IGNORECASE)
    return stem


def frame_key_from_name(path_or_name):
    stem = normalize_image_id(path_or_name)
    match = re.search(r'(vid_res_\d+).*?(frame_\d+)', stem, flags=re.IGNORECASE)
    if match:
        return f'{match.group(1).lower()}_{match.group(2).lower()}'
    return stem.lower()


def relative_project_path(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT)).replace('\\', '/')
    except Exception:
        return str(path).replace('\\', '/')


def image_size(path):
    with Image.open(path) as img:
        return img.size


def build_raw_image_lookup():
    exact = {}
    by_frame_key = {}

    for path in list_images(RAW_IMAGE_DIR):
        exact[normalize_image_id(path.name)] = path
        key = frame_key_from_name(path.name)
        by_frame_key.setdefault(key, []).append(path)

    return exact, by_frame_key


RAW_EXACT_LOOKUP, RAW_FRAME_KEY_LOOKUP = build_raw_image_lookup()


def find_raw_image_for_annotation(annotation_path):
    annotation_id = normalize_image_id(annotation_path.name)

    if annotation_id in RAW_EXACT_LOOKUP:
        return RAW_EXACT_LOOKUP[annotation_id], 'exact_image_id'

    key = frame_key_from_name(annotation_path.name)
    matches = RAW_FRAME_KEY_LOOKUP.get(key, [])

    if len(matches) == 1:
        return matches[0], 'frame_key_match'

    if len(matches) > 1:
        return None, 'ambiguous_frame_key_match'

    return None, 'no_raw_match'


def make_green_mask(image, tolerance):
    arr = np.array(image.convert('RGB')).astype(np.int16)
    target = np.array(hex_to_rgb(GREEN_HEX), dtype=np.int16)
    diff = np.abs(arr - target)

    mask = (
        (diff[:, :, 0] <= tolerance)
        & (diff[:, :, 1] <= tolerance)
        & (diff[:, :, 2] <= tolerance)
    )

    return mask.astype(np.uint8)


def clean_mask(mask):
    if CV2_AVAILABLE:
        kernel = np.ones((3, 3), np.uint8)
        return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    return mask


def bbox_from_mask(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None

    return {
        'x_min': int(xs.min()),
        'y_min': int(ys.min()),
        'x_max': int(xs.max()),
        'y_max': int(ys.max()),
        'green_pixel_count': int(len(xs)),
    }


def bbox_from_largest_component(mask):
    if not CV2_AVAILABLE:
        return bbox_from_mask(mask)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    candidates = []

    for label_id in range(1, num_labels):
        x, y, w, h, area = stats[label_id]
        if area < MIN_GREEN_PIXELS:
            continue

        box_area = max(1, int(w * h))
        candidates.append({
            'x_min': int(x),
            'y_min': int(y),
            'x_max': int(x + w - 1),
            'y_max': int(y + h - 1),
            'green_pixel_count': int(area),
            'box_area': box_area,
            'fill_ratio': float(area / box_area),
        })

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda r: (r['box_area'], r['green_pixel_count']), reverse=True)
    return candidates[0]


def detect_green_box_from_annotation(image_path):
    image = Image.open(image_path).convert('RGB')
    width, height = image.size
    annotation_id = normalize_image_id(image_path.name)
    frame_key = frame_key_from_name(image_path.name)

    best = None
    selected_tolerance = None

    # This is the working method from the earlier notebook: strict mask first, loose mask only as fallback.
    for tolerance in [GREEN_TOLERANCE_STRICT, GREEN_TOLERANCE_LOOSE]:
        mask = make_green_mask(image, tolerance)
        mask = clean_mask(mask)
        box = bbox_from_largest_component(mask)

        if box is not None and box['green_pixel_count'] >= MIN_GREEN_PIXELS:
            best = box
            selected_tolerance = tolerance
            break

    if best is None:
        return {
            'annotation_id': annotation_id,
            'frame_key': frame_key,
            'annotated_filename': image_path.name,
            'status': 'failed',
            'failure_reason': 'green box not detected',
            'annotated_width': width,
            'annotated_height': height,
            'coordinate_source': 'annotation_auto_extract_failed',
            'warning': 'no usable green component detected',
        }

    x_min = max(0, int(best['x_min']))
    y_min = max(0, int(best['y_min']))
    x_max = min(width - 1, int(best['x_max']))
    y_max = min(height - 1, int(best['y_max']))

    box_w = x_max - x_min
    box_h = y_max - y_min
    area_ratio = (box_w * box_h) / max(1, width * height)

    warnings = []
    if area_ratio > 0.60:
        warnings.append('large box area')
    if box_w <= 5 or box_h <= 5:
        warnings.append('very small box')
    if x_min <= 2 or y_min <= 2:
        warnings.append('box touches top or left edge')
    if x_max >= width - 3 or y_max >= height - 3:
        warnings.append('box touches right or bottom edge')

    return {
        'annotation_id': annotation_id,
        'frame_key': frame_key,
        'annotated_filename': image_path.name,
        'status': 'success',
        'failure_reason': '',
        'annotated_width': width,
        'annotated_height': height,
        'source_x_min': x_min,
        'source_y_min': y_min,
        'source_x_max': x_max,
        'source_y_max': y_max,
        'green_pixel_count': best.get('green_pixel_count', ''),
        'box_area_ratio': round(float(area_ratio), 6),
        'selected_tolerance': selected_tolerance,
        'coordinate_source': 'annotation_auto_extract',
        'warning': '; '.join(warnings),
    }


def scale_box_to_raw(row, raw_path, raw_match_method):
    row = dict(row)
    row['raw_match_method'] = raw_match_method
    row['annotated_image_path'] = relative_project_path(ANNOTATED_DIR / row.get('annotated_filename', ''))

    if raw_path is None:
        row['image_id'] = ''
        row['image_path'] = ''
        row['raw_filename'] = ''
        row['status'] = 'failed'
        row['failure_reason'] = (str(row.get('failure_reason', '')) + '; raw residence image not found').strip('; ')
        row['warning'] = (str(row.get('warning', '')) + '; raw residence image not found').strip('; ')
        return row

    row['image_id'] = raw_path.stem
    row['raw_filename'] = raw_path.name
    row['image_path'] = relative_project_path(raw_path)

    raw_w, raw_h = image_size(raw_path)
    row['raw_width'] = raw_w
    row['raw_height'] = raw_h

    if row.get('status') != 'success':
        return row

    src_w = int(row['annotated_width'])
    src_h = int(row['annotated_height'])
    sx = raw_w / src_w
    sy = raw_h / src_h

    row['scale_x_to_raw'] = round(float(sx), 6)
    row['scale_y_to_raw'] = round(float(sy), 6)

    row['gt_x_min'] = int(round(float(row['source_x_min']) * sx))
    row['gt_y_min'] = int(round(float(row['source_y_min']) * sy))
    row['gt_x_max'] = int(round(float(row['source_x_max']) * sx))
    row['gt_y_max'] = int(round(float(row['source_y_max']) * sy))

    row['gt_x_min'] = max(0, min(row['gt_x_min'], raw_w - 1))
    row['gt_y_min'] = max(0, min(row['gt_y_min'], raw_h - 1))
    row['gt_x_max'] = max(0, min(row['gt_x_max'], raw_w - 1))
    row['gt_y_max'] = max(0, min(row['gt_y_max'], raw_h - 1))

    ann_ar = src_w / max(1, src_h)
    raw_ar = raw_w / max(1, raw_h)
    ar_diff = abs(ann_ar - raw_ar) / max(raw_ar, 1e-9)
    row['aspect_ratio_relative_difference'] = round(float(ar_diff), 6)

    if abs(sx - 1.0) > 0.01 or abs(sy - 1.0) > 0.01:
        row['warning'] = (str(row.get('warning', '')) + '; coordinates scaled to raw image size').strip('; ')
    if ar_diff > 0.02:
        row['warning'] = (str(row.get('warning', '')) + '; annotated/raw aspect ratio mismatch').strip('; ')

    if row['gt_x_max'] <= row['gt_x_min'] or row['gt_y_max'] <= row['gt_y_min']:
        row['status'] = 'failed'
        row['failure_reason'] = 'invalid scaled box'

    return row


def draw_ground_truth_review(row, output_path):
    image_path = PROJECT_ROOT / str(row.get('image_path', ''))
    if not image_path.exists():
        return None

    img = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(img)

    x1 = int(row['gt_x_min'])
    y1 = int(row['gt_y_min'])
    x2 = int(row['gt_x_max'])
    y2 = int(row['gt_y_max'])

    green = hex_to_rgb(GREEN_HEX)
    for offset in range(5):
        draw.rectangle([x1 - offset, y1 - offset, x2 + offset, y2 + offset], outline=green)

    draw.rectangle([0, 0, img.width, 48], fill='black')
    label = f"GT {row['image_id']} | {row.get('frame_key', '')} | {row.get('raw_match_method', '')}"
    draw.text((8, 14), label[:180], fill='white')

    output_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(output_path, quality=92)
    return output_path


def make_contact_sheet(image_paths, output_path, thumb_width=360):
    image_paths = [Path(p) for p in image_paths if p and Path(p).exists()]
    if not image_paths:
        return ''

    thumbs = []
    for p in image_paths:
        img = Image.open(p).convert('RGB')
        w, h = img.size
        new_h = max(1, int(h * (thumb_width / w)))
        img = img.resize((thumb_width, new_h), Image.Resampling.LANCZOS)
        thumbs.append((p.name, img))

    cols = 3
    pad = 12
    label_h = 26
    max_h = max(img.height for _, img in thumbs)
    rows = math.ceil(len(thumbs) / cols)

    sheet_w = cols * thumb_width + (cols + 1) * pad
    sheet_h = rows * (max_h + label_h + pad) + pad
    sheet = Image.new('RGB', (sheet_w, sheet_h), 'white')
    draw = ImageDraw.Draw(sheet)

    for idx, (name, img) in enumerate(thumbs):
        r = idx // cols
        c = idx % cols
        x = pad + c * (thumb_width + pad)
        y = pad + r * (max_h + label_h + pad)
        sheet.paste(img, (x, y))
        draw.text((x, y + img.height + 4), name[:48], fill=(0, 0, 0))

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, quality=92)
    return str(output_path)


def filename_to_description(image_id):
    text = str(image_id)
    text = re.sub(r'\.[A-Za-z0-9]+$', '', text)
    text = re.sub(r'[_\-]+', ' ', text)
    text = re.sub(r'\b(annotated|annotation|ground|truth|gt|boxed|with|box|copy|image|img|photo|screenshot|screen|shot)\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(frame|frames)\s*\d*\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b(vid|video|residence|res)\s*\d*\b', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b\d+\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text if text else str(image_id).replace('_', ' ').replace('-', ' ').strip().lower()


def infer_target_object(description):
    text = f' {str(description).lower()} '
    rules = [
        ('elevator button panel', ['elevator button panel', 'button panel', 'elevator panel']),
        ('elevator button', ['elevator button', 'call button']),
        ('package', ['package', 'parcel', 'delivery', 'cardboard box', 'box']),
        ('person', ['person', 'human', 'man', 'woman', 'girl', 'boy']),
        ('door', ['door', 'apartment door']),
        ('chair', ['chair', 'seat']),
        ('table', ['table', 'desk']),
        ('plant', ['plant', 'flower']),
        ('shoes', ['shoes', 'shoe']),
        ('bag', ['bag', 'backpack']),
        ('sign', ['sign', 'label']),
        ('water dispenser', ['water dispenser', 'dispenser']),
        ('elevator', ['elevator', 'lift']),
        ('wall', ['wall']),
        ('lobby', ['lobby']),
        ('lounge', ['lounge']),
        ('hallway', ['hallway', 'corridor']),
    ]
    for target, keywords in rules:
        for keyword in keywords:
            if f' {keyword} ' in text or keyword in text:
                return target
    words = [w for w in re.split(r'\s+', text.strip()) if w]
    return words[0] if words else 'object'


def infer_target_type(target_object):
    if target_object in {'person', 'man', 'woman', 'girl', 'boy', 'human'}:
        return 'person'
    if target_object in {'lobby', 'lounge', 'hallway', 'corridor', 'laundry', 'elevator', 'wall'}:
        return 'place'
    return 'object'


def add_prompt_columns_from_annotation_filename(df):
    df = df.copy()
    df['filename_description'] = df['annotation_id'].map(filename_to_description)
    df['grounding_prompt'] = df['filename_description']
    df['target_object'] = df['filename_description'].map(infer_target_object)
    df['target_label'] = df['target_object']
    df['target_type'] = df['target_object'].map(infer_target_type)
    df['command'] = df['grounding_prompt'].map(lambda x: f'find {x}' if str(x).strip() else 'find target object')
    df['prompt_source'] = 'generated_from_current_ground_truth_annotation_filename'
    return df


In [ ]:

# Generate ground truth coordinates and confirmation screenshots.

if not ANNOTATED_DIR.exists():
    raise FileNotFoundError(f'Annotated folder not found: {ANNOTATED_DIR}')
if not RAW_IMAGE_DIR.exists():
    raise FileNotFoundError(f'Raw residence image folder not found: {RAW_IMAGE_DIR}')

annotation_images = list_images(ANNOTATED_DIR)
raw_images = list_images(RAW_IMAGE_DIR)

if not annotation_images:
    raise ValueError(f'No annotated images found in: {ANNOTATED_DIR}')
if not raw_images:
    raise ValueError(f'No raw residence images found in: {RAW_IMAGE_DIR}')

# Clear old review screenshots so this folder reflects the current run only.
if FINAL_REVIEW_SCREENSHOT_DIR.exists():
    shutil.rmtree(FINAL_REVIEW_SCREENSHOT_DIR)
FINAL_REVIEW_SCREENSHOT_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for annotation_path in annotation_images:
    detected = detect_green_box_from_annotation(annotation_path)
    raw_path, match_method = find_raw_image_for_annotation(annotation_path)
    scaled = scale_box_to_raw(detected, raw_path, match_method)
    rows.append(scaled)

full_df = pd.DataFrame(rows).sort_values(['frame_key', 'annotation_id']).reset_index(drop=True)

# Defensive cleanup so every generated row has the expected fields.
for col, default in {
    'status': 'failed',
    'image_path': '',
    'failure_reason': '',
    'raw_match_method': '',
}.items():
    if col not in full_df.columns:
        full_df[col] = default

for col in ['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']:
    if col not in full_df.columns:
        full_df[col] = np.nan
    full_df[col] = pd.to_numeric(full_df[col], errors='coerce')

valid_box = (
    full_df['status'].eq('success')
    & full_df.get('image_path', '').fillna('').astype(str).ne('')
    & full_df[['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']].notna().all(axis=1)
    & (full_df['gt_x_max'] > full_df['gt_x_min'])
    & (full_df['gt_y_max'] > full_df['gt_y_min'])
)

success_df = full_df[valid_box].copy()
failed_df = full_df[~valid_box].copy()

if success_df.empty:
    debug_path = OUTPUT_DIR / 'debug_full_ground_truth_rows_no_success.csv'
    full_df.to_csv(debug_path, index=False)
    raise RuntimeError(f'No successful ground truth boxes were generated. Debug rows saved to: {debug_path}')

# Final ground truth columns.
gt_columns = [
    'image_id',
    'frame_key',
    'annotation_id',
    'image_path',
    'annotated_image_path',
    'raw_filename',
    'annotated_filename',
    'raw_match_method',
    'gt_x_min',
    'gt_y_min',
    'gt_x_max',
    'gt_y_max',
    'raw_width',
    'raw_height',
    'annotated_width',
    'annotated_height',
    'source_x_min',
    'source_y_min',
    'source_x_max',
    'source_y_max',
    'scale_x_to_raw',
    'scale_y_to_raw',
    'green_pixel_count',
    'box_area_ratio',
    'selected_tolerance',
    'coordinate_source',
    'aspect_ratio_relative_difference',
    'warning',
]

for col in gt_columns:
    if col not in success_df.columns:
        success_df[col] = ''

for col in ['gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max', 'raw_width', 'raw_height', 'annotated_width', 'annotated_height', 'source_x_min', 'source_y_min', 'source_x_max', 'source_y_max']:
    if col in success_df.columns:
        success_df[col] = pd.to_numeric(success_df[col], errors='coerce').fillna(0).astype(int)

gt_df = success_df[gt_columns].copy().sort_values(['frame_key', 'image_id']).reset_index(drop=True)

# Write main coordinate and audit files.
gt_df.to_csv(GT_CSV, index=False)
failed_df.to_csv(SKIPPED_CSV, index=False)
full_df.to_csv(MATCH_AUDIT_CSV, index=False)

# Confirmation screenshots are drawn on raw residence images using final coordinates.
review_paths = []
for _, row in gt_df.iterrows():
    output_path = FINAL_REVIEW_SCREENSHOT_DIR / f"{row['frame_key']}_{row['image_id']}_ground_truth_confirmation.jpg"
    review_path = draw_ground_truth_review(row, output_path)
    if review_path:
        review_paths.append(review_path)

make_contact_sheet(review_paths, CONTACT_SHEET_PATH)

print('Ground truth coordinate generation complete')
print(f'Annotated images processed: {len(full_df)}')
print(f'Raw residence images available: {len(raw_images)}')
print(f'Successful coordinate rows: {len(gt_df)}')
print(f'Skipped or failed rows: {len(failed_df)}')
print('')
print('Raw match methods:')
display(
    gt_df['raw_match_method']
    .value_counts(dropna=False)
    .rename_axis('raw_match_method')
    .reset_index(name='rows')
)
print('')
print(f'Saved coordinate CSV: {GT_CSV}')
print(f'Saved skipped/problem CSV: {SKIPPED_CSV}')
print(f'Saved match audit CSV: {MATCH_AUDIT_CSV}')
print(f'Saved confirmation screenshots: {FINAL_REVIEW_SCREENSHOT_DIR}')
print(f'Saved contact sheet: {CONTACT_SHEET_PATH}')
print('')

display(gt_df.head(20))

warning_df = gt_df[gt_df['warning'].fillna('').astype(str).str.len() > 0].copy()
print(f'Rows with warnings: {len(warning_df)}')
if len(warning_df) > 0:
    display(warning_df[['image_id', 'frame_key', 'raw_match_method', 'warning', 'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max']])

if len(failed_df) > 0:
    print('Skipped/problem rows:')
    display(failed_df[['annotation_id', 'frame_key', 'status', 'failure_reason', 'raw_match_method']].head(30))


In [ ]:

# Create the benchmark used by Notebook 03.
# This replaces the obsolete day3_selected_frames.csv.

benchmark_df = add_prompt_columns_from_annotation_filename(gt_df)

benchmark_columns = [
    'image_id',
    'frame_key',
    'annotation_id',
    'image_path',
    'annotated_image_path',
    'grounding_prompt',
    'target_object',
    'target_label',
    'target_type',
    'command',
    'filename_description',
    'prompt_source',
    'gt_x_min',
    'gt_y_min',
    'gt_x_max',
    'gt_y_max',
    'raw_width',
    'raw_height',
    'annotated_width',
    'annotated_height',
    'source_x_min',
    'source_y_min',
    'source_x_max',
    'source_y_max',
    'scale_x_to_raw',
    'scale_y_to_raw',
    'green_pixel_count',
    'box_area_ratio',
    'selected_tolerance',
    'coordinate_source',
    'aspect_ratio_relative_difference',
    'warning',
]

for col in benchmark_columns:
    if col not in benchmark_df.columns:
        benchmark_df[col] = ''

benchmark_df = benchmark_df[benchmark_columns].sort_values(['frame_key', 'image_id']).reset_index(drop=True)

benchmark_df.to_csv(GROUNDING_BENCHMARK_CSV, index=False)

print('New benchmark creation complete')
print(f'Approved ground truth rows used: {len(gt_df)}')
print(f'New benchmark rows written: {len(benchmark_df)}')
print('')
print(f'Saved new grounding benchmark CSV: {GROUNDING_BENCHMARK_CSV}')
print('day3_selected_frames.csv was not read and was not overwritten.')
print('')
print('Prompt source: generated from the current ground truth annotation filenames.')
print('Review the grounding_prompt and target_object columns before running Notebook 03 if a filename is not descriptive enough.')

display(benchmark_df[[
    'image_id',
    'frame_key',
    'annotation_id',
    'image_path',
    'grounding_prompt',
    'target_object',
    'gt_x_min',
    'gt_y_min',
    'gt_x_max',
    'gt_y_max',
]].head(40))


In [ ]:
# Notebook 02 output contract for Notebooks 03 and 04.

required_gt_columns = {
    'image_id', 'image_path',
    'gt_x_min', 'gt_y_min', 'gt_x_max', 'gt_y_max',
}
required_benchmark_columns = {
    'image_id', 'image_path', 'grounding_prompt',
    'target_object', 'target_type', 'command',
}

missing_gt_columns = required_gt_columns - set(gt_df.columns)
missing_benchmark_columns = required_benchmark_columns - set(benchmark_df.columns)

if missing_gt_columns:
    raise ValueError(f'ground_truth_boxes.csv is missing columns: {sorted(missing_gt_columns)}')
if missing_benchmark_columns:
    raise ValueError(
        'grounding_benchmark_from_ground_truth.csv is missing columns: '
        f'{sorted(missing_benchmark_columns)}'
    )

gt_ids = set(gt_df['image_id'].astype(str))
benchmark_ids = set(benchmark_df['image_id'].astype(str))
if gt_ids != benchmark_ids:
    raise ValueError(
        'Notebook 02 output ID mismatch. '
        f'GT-only={sorted(gt_ids - benchmark_ids)[:10]}, '
        f'benchmark-only={sorted(benchmark_ids - gt_ids)[:10]}'
    )

missing_raw_paths = []
for image_path_value in benchmark_df['image_path'].astype(str):
    path = Path(image_path_value)
    resolved = path if path.is_absolute() else PROJECT_ROOT / path
    if not resolved.exists():
        missing_raw_paths.append(str(resolved))

if missing_raw_paths:
    raise FileNotFoundError(
        'Benchmark references raw images that do not exist:\n'
        + '\n'.join(missing_raw_paths[:20])
    )

for required_output in [GT_CSV, GROUNDING_BENCHMARK_CSV]:
    if not required_output.exists():
        raise FileNotFoundError(f'Notebook 02 did not create: {required_output}')

print('Notebook 02 integration contract passed.')
print('Notebook 03 reads:', GROUNDING_BENCHMARK_CSV)
print('Notebook 03 and 04 read:', GT_CSV)
print('Notebook 03 writes parsed prompts to:', PARSED_BENCHMARK_CSV)


## Files used and generated

Reads:

```text
examples/ground_truth_annotated/
examples/residence_images/
```

Does not read:

```text
examples/day3_selected_frames.csv
examples/manual_ground_truth_overrides.csv
```

Writes:

```text
examples/ground_truth_boxes.csv
examples/grounding_benchmark_from_ground_truth.csv
results/ground_truth_coordinate_update/final_ground_truth_review_screenshots/
results/ground_truth_coordinate_update/final_ground_truth_review_contact_sheet.jpg
results/ground_truth_coordinate_update/skipped_ground_truth_images.csv
results/ground_truth_coordinate_update/annotation_to_raw_image_match_audit.csv
```
